# Maia Corpus v8 — empacotamento incremental e retomável (CPU)

Monte o Drive. Execute primeiro o dry-run, depois 1 pacote piloto. O script usa ZIP_STORED, verifica SHA-256 por shard, envia cada ZIP concluído ao Drive e cria manifesto. Não execute em paralelo em duas sessões. O limite de 22 horas é checado entre pacotes; um pacote em andamento pode ultrapassar o limite. Verifique os hashes dos ZIPs após baixá-los para SSD antes de treinar.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
p=Path("/content/maia_corpus_pack_colab.py")
p.write_text('#!/usr/bin/env python3\n"""Maia Corpus v8: resumable Drive -> local ZIP -> Drive packer (CPU only)."""\nimport argparse, hashlib, json, os, shutil, sqlite3, sys, time, zipfile\nfrom pathlib import Path\n\nROOT_REL = \'Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full\'\nDB_REL = \'recovery_audit_v4/state_operational_v5.sqlite3\'\nBLOCK_BYTES = 2048 * 4\n\ndef digest(path, chunk=4*1024*1024):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        while True:\n            b=f.read(chunk)\n            if not b: break\n            h.update(b)\n    return h.hexdigest()\n\ndef source_path(root, value, source, split, idx):\n    # Catalog path is authoritative; accept paths from the original Colab environment.\n    p=Path(value)\n    if p.is_absolute():\n        s=str(p)\n        marker=\'/pretraining_production_v1_full/\'\n        if marker in s:\n            p=root / s.split(marker,1)[1]\n        elif p.exists():\n            return p\n        else:\n            raise ValueError(f\'Unexpected absolute catalog path: {value}\')\n    else:\n        if p.parts and p.parts[0]==\'shards\': p=root/p\n        else: p=root/\'shards\'/p\n    if not p.is_file():\n        fallback=root/\'shards\'/source/split/f\'{source}_{split}_{idx:06d}.bin\'\n        if fallback.is_file(): return fallback\n        raise FileNotFoundError(str(p))\n    return p\n\ndef plan(rows, max_bytes):\n    packs=[]; current=[]; total=0\n    for r in rows:\n        size=r[4]*BLOCK_BYTES\n        if current and total+size>max_bytes:\n            packs.append(current);current=[];total=0\n        current.append(r);total+=size\n    if current: packs.append(current)\n    return packs\n\ndef write_json_atomic(path, data):\n    tmp=path.with_name(path.name+\'.partial\')\n    with open(tmp,\'w\',encoding=\'utf-8\') as f:\n        json.dump(data,f,ensure_ascii=False,indent=2,sort_keys=True)\n        f.flush();os.fsync(f.fileno())\n    os.replace(tmp,path)\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--drive-root\',default=\'/content/drive/MyDrive\')\n    ap.add_argument(\'--local-root\',default=\'/content/maia_corpus_pack_work\')\n    ap.add_argument(\'--pack-gib\',type=float,default=1.0)\n    ap.add_argument(\'--hours\',type=float,default=22.0)\n    ap.add_argument(\'--max-packs\',type=int,default=0)\n    ap.add_argument(\'--dry-run\',action=\'store_true\')\n    args=ap.parse_args()\n    if args.pack_gib<=0 or args.hours<=0: ap.error(\'pack-gib and hours must be positive\')\n    root=Path(args.drive_root)/ROOT_REL\n    db=root/DB_REL\n    dest=root/\'packed_zip_v1\'\n    local=Path(args.local_root)\n    if not db.exists(): raise FileNotFoundError(f\'Missing catalog: {db}\')\n    # Query only; never run PRAGMA quick_check on mounted Drive.\n    uri=db.as_uri()+\'?mode=ro\'\n    con=sqlite3.connect(uri,uri=True,timeout=30)\n    try:\n        rows=con.execute(\'SELECT source,split,shard_index,path,blocks,sha256 FROM shard ORDER BY source,split,shard_index\').fetchall()\n    finally: con.close()\n    assert len(rows)==60612, f\'Unexpected shard count: {len(rows)}\'\n    assert sum(r[4] for r in rows)==3607119, \'Unexpected block count\'\n    # Fixed grouping by metadata, stable across restarts.\n    packs=plan(rows,int(args.pack_gib*(1024**3)))\n    print(f\'Shards: {len(rows):,}; blocks: {sum(r[4] for r in rows):,}; ZIP packs: {len(packs)}\',flush=True)\n    if args.dry_run: return\n    dest.mkdir(parents=True,exist_ok=True)\n    local.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic(); completed=0; skipped=0\n    for i, group in enumerate(packs):\n        if time.monotonic()-started >= args.hours*3600:\n            print(\'Time budget reached. Safe stop before next pack.\',flush=True);break\n        source,split=group[0][0:2]\n        # Pack names deterministic, independent of session.\n        name=f\'pack_{i:04d}_{source}_{split}.zip\'\n        remote=dest/name; meta=dest/(name+\'.json\')\n        expected_bytes=sum(r[4]*BLOCK_BYTES for r in group)\n        if remote.exists() and meta.exists():\n            try:\n                info=json.loads(meta.read_text(encoding=\'utf-8\'))\n                if (info.get(\'status\')==\'complete\' and info.get(\'zip_bytes\')==remote.stat().st_size\n                    and info.get(\'raw_bytes\')==expected_bytes and info.get(\'count\')==len(group)\n                    and info.get(\'first\')==list(group[0][:3]) and info.get(\'last\')==list(group[-1][:3])):\n                    skipped+=1\n                    print(f\'[{i+1}/{len(packs)}] REUSE {name}\',flush=True)\n                    continue\n            except Exception as e: print(f\'Cannot reuse {name}: {e}\',flush=True)\n            raise RuntimeError(f\'Incomplete or conflicting destination pack: {name}; inspect before retry\')\n        if remote.exists() or meta.exists():\n            raise RuntimeError(f\'Orphaned ZIP/manifest for {name}; inspect before retry\')\n        free=shutil.disk_usage(local).free\n        if free < expected_bytes + 2*(1024**3):\n            raise RuntimeError(f\'Insufficient local disk for {name}: free={free}, needed~{expected_bytes+2*(1024**3)}\')\n        temp=local/(name+\'.partial\');temp.unlink(missing_ok=True)\n        print(f\'[{i+1}/{len(packs)}] BUILD {name} shards={len(group)} rawGiB={expected_bytes/1024**3:.3f}\',flush=True)\n        t=time.monotonic()\n        try:\n            with zipfile.ZipFile(temp,\'w\',compression=zipfile.ZIP_STORED,allowZip64=True) as z:\n                for j,(src,sp,idx,path,blocks,sha) in enumerate(group,1):\n                    p=source_path(root,path,src,sp,idx)\n                    size=blocks*BLOCK_BYTES\n                    if p.stat().st_size!=size: raise ValueError(f\'Bad shard size: {p}\')\n                    arc=f\'{src}/{sp}/{src}_{sp}_{idx:06d}.bin\'\n                    h=hashlib.sha256()\n                    with open(p,\'rb\') as inp, z.open(arc,\'w\',force_zip64=True) as out:\n                        count=0\n                        while True:\n                            buf=inp.read(4*1024*1024)\n                            if not buf:break\n                            out.write(buf);h.update(buf);count+=len(buf)\n                    if count!=size or h.hexdigest().lower()!=sha.lower():\n                        raise ValueError(f\'Shard SHA256/size mismatch: {p}\')\n                    if j%100==0 or j==len(group):\n                        print(f\'  {j}/{len(group)} shards | {time.monotonic()-t:.0f}s\',flush=True)\n            # Verify ZIP central directory and contents integrity locally.\n            with zipfile.ZipFile(temp,\'r\') as z:\n                if len(z.infolist())!=len(group) or z.testzip() is not None:\n                    raise ValueError(\'ZIP integrity check failed\')\n            zhash=digest(temp)\n            zsize=temp.stat().st_size\n            # Copy completed ZIP to Drive using staging name, then rename.\n            staged=dest/(name+\'.uploading\')\n            staged.unlink(missing_ok=True)\n            try:\n                shutil.copyfile(temp,staged)\n                if staged.stat().st_size!=zsize:\n                    raise IOError(\'Remote staged ZIP size mismatch\')\n                # Avoid rereading remote ZIP (can be very slow on Drive mount).\n                os.replace(staged,remote)\n            except Exception:\n                print(f\'Upload failed; staged file may remain: {staged}\',flush=True)\n                raise\n            info={\'status\':\'complete\',\'zip_name\':name,\'zip_bytes\':zsize,\'zip_sha256\':zhash,\n                  \'raw_bytes\':expected_bytes,\'count\':len(group),\n                  \'first\':list(group[0][:3]),\'last\':list(group[-1][:3]),\n                  \'sha256_verified_per_shard\':True,\'format\':\'zip-stored-v1\',\n                  \'catalog_shards\':len(rows),\'catalog_blocks\':sum(r[4] for r in rows)}\n            write_json_atomic(meta,info)\n            temp.unlink(missing_ok=True)\n            completed+=1\n            print(f\'  SAVED {name} {zsize/1024**3:.3f} GiB sha256={zhash} elapsed={time.monotonic()-t:.1f}s\',flush=True)\n            if args.max_packs and completed>=args.max_packs:\n                print(\'max-packs reached\',flush=True);break\n        except Exception:\n            print(f\'FAILED {name}; local partial preserved for diagnosis: {temp}\',flush=True)\n            raise\n    print(f\'DONE this run: completed={completed} reused={skipped}; total packs={len(packs)}\',flush=True)\n    print(\'IMPORTANT: remote ZIP SHA256 must be verified after downloading to SSD before training.\',flush=True)\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")
print("Script criado:",p)

In [ ]:
!python /content/maia_corpus_pack_colab.py --dry-run

## Teste: um pacote
Descomente e execute a célula abaixo após conferir o dry-run.

In [ ]:
# !python /content/maia_corpus_pack_colab.py --max-packs 1 --hours 22

## Migração
Execute somente após o piloto. Em uma nova sessão, execute novamente as células anteriores: os pacotes com manifesto completo serão reutilizados.

In [ ]:
# !python /content/maia_corpus_pack_colab.py --hours 22